In [12]:
# 📓 Range-Doppler Modeli - M2 Macbook Uyumlu (CPU ile Çalışır, MPS Hatası Yok)

# ✅ Hücre 1: Gerekli importlar ve cihaz ayarı (Sadece CPU kullanılır)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from fpdf import FPDF
from datetime import datetime

# Apple M1/M2 uyumu için GPU yerine CPU kullanılacak
device = "cpu"
print(f"✅ Kullanılan cihaz: {device.upper()} (MPS uyumsuzluk önlendi)")


✅ Kullanılan cihaz: CPU (MPS uyumsuzluk önlendi)


In [13]:
# ✅ Hücre 2: Veri seti sınıfı
class RangeDopplerDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []
        self.labels = []
        for label in ['target', 'no_target']:
            class_dir = os.path.join(root_dir, label)
            for file in os.listdir(class_dir):
                if file.endswith('.png') or file.endswith('.jpg'):
                    self.image_files.append(os.path.join(class_dir, file))
                    self.labels.append(1 if label == 'target' else 0)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)


In [14]:
# ✅ Hücre 3: Hafifletilmiş Custom CNN modeli (AdaptiveAvgPool kaldırıldı, M2 uyumlu)
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=8)  # sabit havuzlama
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 24, 256),  # 36864 giriş!
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [15]:
# ✅ Hücre 4: Dönüşüm ve veri yükleme
# ✅ Eğitim dönüşümleri
train_transform = transforms.Compose([
    transforms.Resize((562, 2318)),  # Sabit boyut (yükseklik, genişlik)
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandomAffine(10, translate=(0.1,0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ✅ Test dönüşümleri
test_transform = transforms.Compose([
    transforms.Resize((562, 2318)),  # Sabit boyut
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ✅ Dataset tanımları
train_dataset = RangeDopplerDataset("data/train", transform=train_transform)
test_dataset = RangeDopplerDataset("data/test", transform=test_transform)

# ✅ DataLoader tanımları
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:
# ✅ Hücre 5: Modeli eğit, en iyi modeli kaydet (models klasörünü oluşturur)
os.makedirs("models", exist_ok=True)
model = CustomCNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
criterion = nn.BCELoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

def train_model(model, train_loader, test_loader, optimizer, criterion, scheduler=None, device='cpu', epochs=10, save_path="models/best_model.pth"):
    model.train()
    best_accuracy = 0.0
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if scheduler:
            scheduler.step()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")
        accuracy = evaluate_model(model, test_loader, device)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), save_path)
            print(f"✅ Yeni en iyi model kaydedildi: {best_accuracy:.4f}")

def evaluate_model(model, test_loader, device='cpu'):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze()
            preds = (outputs > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = sum([1 for i in range(len(all_preds)) if all_preds[i]==all_labels[i]]) / len(all_preds)
    print(f"Accuracy: {acc:.4f}")
    return acc

train_model(model, train_loader, test_loader, optimizer, criterion, scheduler, device=device, epochs=10)


Epoch 1, Loss: 0.2018
Accuracy: 0.9700
✅ Yeni en iyi model kaydedildi: 0.9700


In [ ]:
# ✅ Hücre 6: Tahmin fonksiyonu

def predict(image_path, model_path="models/best_model.pth", device='cpu'):
    model = CustomCNN()
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.to(device)
    model.eval()
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image).squeeze()
        return "Hedef Var ✅" if output > 0.5 else "Hedef Yok ❌"


In [ ]:
# ✅ Hücre 7: Tahmin Örneği
image_path = "data/test/target/Zaz - Je veux (Studio version, HD) (128kbit_AAC)Part001.png"  # Örneği değiştir
print(f"{image_path} için tahmin sonucu: {predict(image_path, device=device)}")
